# Грунт против «масложора»

Два участника на одних и тех же данных и одних и тех же тестах:

- **ученик** (`acquire.py` + `core.py`): общие схемы рассуждения без единого слова, язык задач выучивает сам по рассказам с ответами, с 20 рассказов на задачу. Считает на процессоре;
- **обычный трансформер** с нуля (`colab/baseline.py`): тот же текст на входе, ответ на выходе; ему даются и те же 20 рассказов, и в 10–100 раз больше. Считает на видеокарте.

Четыре набора: 20 задач bAbI; 12 новых задач A и 12 задач B (по ним грунт дорабатывался); 12 новых задач C (отложенные: их писали другие авторы, код до них был заморожен и запущен на них один раз).

И отдельно — свободный язык (этап 7): 17 задач, пересказанных другими словами. Ученик на них один даёт 0 %; читалка — маленькая сеть, которую учит ученик и проверяет грунт, — читает их по 20 шаблонным и 40 свободным рассказам. Трансформер получает те же 60 рассказов.

Как запускать:

1. Среда выполнения → Сменить среду → T4 (или L4) и «Большой объём ОЗУ» (там больше процессоров).
2. Среда выполнения → Выполнить всё. Идёт около четырёх-пяти часов: трансформер учится в фоне, пока считает ученик.
3. Если оборвалось, выполнить всё снова: продолжит с места остановки.
4. Нужен вывод последней ячейки (таблицы между линиями из `=`).


In [ ]:
import os, subprocess, sys
WORK = '/content/mx-mind'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/mx-mind'
except Exception as e:
    print('Диск не подключён, итог останется в', WORK, '-', e)
os.makedirs(WORK, exist_ok=True)
if os.path.exists('/content/MX-Project'):
    subprocess.run(['git', '-C', '/content/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', '/content/MX-Project'], check=True)
M5 = '/content/MX-Project/research_m5'
os.environ['PYTHONHASHSEED'] = '0'
!python -u {M5}/colab/mind.py --work {WORK} --fetch
!python {M5}/newtasks.py --set A --out {WORK}/new/A | tail -1
!python {M5}/newtasks.py --set B --out {WORK}/new/B | tail -1
!python {M5}/newtasks.py --set C --out {WORK}/new/C | tail -1


## Трансформер (в фоне)

bAbI: те же 20 рассказов на задачу, стандартный набор (1000 вопросов на задачу) и большой (10 000). Новые задачи: 20, 200 и 2000 рассказов на задачу. Ход пишется в `baseline.txt` в рабочей папке. Свободный язык: 20 шаблонных и 40 свободных рассказов на задачу, три группы задач.


In [ ]:
B = sys.executable + ' -u ' + M5 + '/colab/baseline.py'
cmd = (f"{B} --data {WORK}/tasks_1-20_v1-2 --lang en --sizes 20,1k,10k --out {WORK}/baseline_en.json; "
       f"{B} --data {WORK}/new/A --lang '' --sizes 20,200,2000 --out {WORK}/baseline_A.json; "
       f"{B} --data {WORK}/new/B --lang '' --sizes 20,200,2000 --out {WORK}/baseline_B.json; "
       f"{B} --data {WORK}/new/C --lang '' --sizes 20,200,2000 --out {WORK}/baseline_C.json")
!python {M5}/free/run.py --babi {WORK}/tasks_1-20_v1-2 --new {WORK}/new --out {WORK}/free --plain {WORK}/free_plain --no-reader | tail -1
cmd += ''.join(f"; {B} --data {WORK}/free_plain/{g} --lang '' --sizes 60 --out {WORK}/baseline_free_{g}.json" for g in ('dev', 'held_out_1', 'held_out_2'))
base = subprocess.Popen(cmd, shell=True, stdout=open(WORK + '/baseline.txt', 'w'), stderr=subprocess.STDOUT)
print('трансформер запущен в фоне')


## Ученик: новые задачи

Каждая задача учится отдельно и с нуля, с 20 рассказов.

In [ ]:
T = '1,2,3,4,5,6,7,8,9,10,11,12'
for name in 'ABC':
    out = f'{WORK}/learner_{name}.txt'
    if not os.path.exists(out) or 'all:' not in open(out).read():
        !python -u {M5}/acquire.py --data {WORK}/new/{name} --stories 20 --apart --tasks {T} > {out}
    print('набор', name); print(''.join(l for l in open(out) if l.startswith(('test', 'all'))))


## Ученик: bAbI

Английский, хинди и английский с заменёнными буквами (шифр) по 20 рассказов на задачу; для английского ещё 10 и 50 рассказов. Раз в пять минут печатает, сколько задач пройдено.

In [ ]:
!python -u {M5}/colab/mind.py --work {WORK}


## Свободный язык (этап 7)

Для каждой из 17 задач: ученик учит шаблонный язык по 20 рассказам, читалка учится читать пересказ. В таблице — доля верных ответов на свободном тесте: ученик один, читалка без свободных рассказов, читалка с 40 свободными рассказами с ответами, и лучшее, что вообще допускают её кандидаты. Группы: `dev` — по этим задачам читалка писалась; `held-out 1` — отложенные для первого круга; `held-out 2` — отложенные для второго (нынешнего) кода.


In [ ]:
!python -u {M5}/free/run.py --babi {WORK}/tasks_1-20_v1-2 --new {WORK}/new --out {WORK}/free --vectors get --jobs {os.cpu_count()} | tee {WORK}/free.txt


## Итог

In [ ]:
try:
    base.wait()
except NameError:
    pass
print('=' * 70)
print('УЧЕНИК, bAbI')
print(open(WORK + '/results.md').read())
for name in 'ABC':
    print('УЧЕНИК, новые задачи', name, '(20 рассказов):', [l.strip() for l in open(f'{WORK}/learner_{name}.txt') if l.startswith('all')])
print('=' * 70)
print('СВОБОДНЫЙ ЯЗЫК: читалка (17 задач)')
print(open(WORK + '/free.txt').read())
print('=' * 70)
print('ТРАНСФОРМЕР (по порядку: bAbI, новые A, новые B, новые C, свободный язык: dev, held-out 1, held-out 2)')
lines = open(WORK + '/baseline.txt').read().splitlines()
print('\n'.join(l for l in lines if l.startswith('|') or 'test' in l) or '\n'.join(lines[-15:]))
print('=' * 70)
